In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data splitting and validation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Feature selection (Methodology 3.4 - Feature Selection)
from sklearn.feature_selection import SelectFromModel


Academic Data

In [2]:
import os
print(os.getcwd())

c:\Users\bayibayik\OneDrive - Cape Peninsula University of Technology\Work Documents\FTE Dashboard and Data


In [3]:
import pandas as pd

path = r"C:\Users\bayibayik\OneDrive - Cape Peninsula University of Technology\Work Documents\FTE Dashboard and Data\Hemis_data\Hemis_data\Academic\Academic records 2025-1 - Copy.xlsx"

df_acad = pd.read_excel(path)

In [4]:
df_acad= df_acad[(df_acad["Period of Study"]== 1) & 
                 (df_acad["Qualification Description"]== "DIPLOMA IN CIVIL ENGINEERING")]
#df_acad.head()

In [5]:
df_acad.columns.tolist()

['Student_Number',
 'Surname',
 'Names',
 'Year',
 'Qualification Code',
 'Qualification Description',
 'Qualification block',
 'Subject_Code',
 'Subject_Description',
 'Subject Block',
 'Subject Offering Type',
 'Offering Type Desc',
 'Period of Study',
 'Qualification Type',
 'Qualification Type Name',
 'Canceldate',
 'Percentage',
 'Result_Code',
 'Result Description']

In [6]:
import hashlib
import pandas as pd

def hash_student_id(student_no):
    if pd.isna(student_no):
        return None

    # Standardise the student number
    student_no = str(student_no).strip()

    # Remove trailing .0 from Excel imports
    if student_no.endswith(".0"):
        student_no = student_no[:-2]

    return hashlib.sha256(student_no.encode("utf-8")).hexdigest()

df_acad["Student_ID"] = df_acad["Student_Number"].apply(hash_student_id)

In [7]:
df_acad.drop(columns=["Student_Number","Surname","Names",'Qualification Description',
 'Qualification block','Qualification Code'], inplace= True)

In [8]:
df_acad.groupby([ "Period of Study"])["Student_ID"].nunique().reset_index(name="unique_students")

,Period of Study,unique_students
0,1,578


In [9]:
df_acad.groupby("Year")["Student_ID"].nunique().reset_index(name="unique_students")

,Year,unique_students
0,2018,22
1,2019,43
2,2020,73
3,2021,63
4,2022,90
5,2023,151
6,2024,193
7,2025,219
8,2026,77


In [10]:
df_acad.isnull().sum()

Year                          0
Subject_Code                  0
Subject_Description           0
Subject Block                 0
Subject Offering Type         0
Offering Type Desc            0
Period of Study               0
Qualification Type            0
Qualification Type Name       0
Canceldate                 6563
Percentage                  243
Result_Code                 243
Result Description          243
Student_ID                    0
dtype: int64

In [11]:
# rename subject description to subject name.
df_acad = df_acad.rename(columns={'Subject_Description': 'SUBJECTNAME'})

In [12]:
# Pivot subjects into columns per student per academic year
df_pivot = df_acad.pivot_table(
    index=['Student_ID', 'Year'],
    columns='SUBJECTNAME',
    values='Percentage',
    aggfunc='first'
)

# Student details per academic year
student_info = (
    df_acad[['Student_ID', 'Year', 'Period of Study']]
    .drop_duplicates(subset=['Student_ID', 'Year'])
    .set_index(['Student_ID', 'Year'])
)


# Combine everything
df_acad_final = (
    student_info
    .join(df_pivot)
    .reset_index()
)

df_acad_final

,Student_ID,Year,Period of Study,APPLIED MECHANICS,COMPUTER SKILLS,CONSTRUCTION MATERIALS TECHNOLOGY,ENGINEERING CHEMISTRY,ENGINEERING DRAWING 1,ENGINEERING MATHEMATICS 1,ENGINEERING PHYSICS,PROFESSIONAL AND TECHNICAL COMMUNICATION,SURVEYING,THEORY OF STRUCTURES
0,47082b09ee3cafe8bd557655d1f09313975a4ba9b5b722...,2018,1,36.0,51.0,74.0,43.0,67.0,23.0,44.0,62.0,53.0,NaN
1,47082b09ee3cafe8bd557655d1f09313975a4ba9b5b722...,2019,1,NaN,NaN,NaN,38.0,NaN,29.0,36.0,NaN,NaN,34.0
2,47082b09ee3cafe8bd557655d1f09313975a4ba9b5b722...,2020,1,NaN,NaN,NaN,50.0,NaN,72.0,72.0,NaN,NaN,55.0
3,2c6a53cd955f47e79ef20a03c53584124bdf09f6a46537...,2018,1,36.0,50.0,65.0,54.0,58.0,65.0,50.0,57.0,60.0,57.0
4,2fdae0754e821aeeefb29e8ceec025136ba8be00fb7b05...,2018,1,27.0,50.0,62.0,63.0,50.0,32.0,55.0,63.0,56.0,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
926,5a10048a9d29db813f210eb701b975e1b70a699e9eb71e...,2025,1,57.0,67.0,57.0,69.0,53.0,69.0,75.0,75.0,57.0,55.0
927,70ed046986645707e6536f00ab89a1f7ad00109dec6d53...,2026,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
928,3c94275d81284fb4b20498f26c3e341f55b4deb67284ba...,2025,1,50.0,86.0,71.0,66.0,69.0,63.0,54.0,80.0,51.0,53.0
929,ec1a93f5f6c7061a5b3e8d1d1732db4c52f13fcbf8afa1...,2025,1,50.0,85.0,82.0,50.0,73.0,57.0,59.0,75.0,71.0,50.0


In [13]:
subject_cols = [
    'APPLIED MECHANICS',
    'COMPUTER SKILLS',
    'CONSTRUCTION MATERIALS TECHNOLOGY',
    'ENGINEERING CHEMISTRY',
    'ENGINEERING DRAWING 1',
    'ENGINEERING MATHEMATICS 1',
    'ENGINEERING PHYSICS',
    'PROFESSIONAL AND TECHNICAL COMMUNICATION',
    'SURVEYING',
    'THEORY OF STRUCTURES'
]

In [14]:
students_no_subjects = df_acad_final[
    df_acad_final[subject_cols].isna().all(axis=1)
]

print("Students with no subject records:", len(students_no_subjects))

students_no_subjects[
    ["Student_ID","Year", "Period of Study"]
]

Students with no subject records: 75


,Student_ID,Year,Period of Study
291,103dd45c302609c5ca2c7afe652325f18f557bc145c720...,2026,1
332,3db997f9461d858f2c34a4350eead7e326d44b202009af...,2026,1
356,d9b342d37bf5c6afc25c6b24d14ddeb1aad43808861d00...,2026,1
397,a69359709995cc15e6917d4cdbc0a30052d52068714d92...,2026,1
416,74894efea8cab788be088bc7913083ce1fa70a4f0d8cbd...,2026,1
...,...,...,...
915,5da75c23f6ce8f9351a8ac86143c5d6b09a71437601417...,2026,1
917,15badadd872799b06eb099672991942d5fa41c20d5650a...,2026,1
923,6e19abba682470d75da71dc2161e90713e6d987a133bf5...,2026,1
925,6bcec1ed0df9242b0e992fee6c186bfce9f091c317cd0b...,2026,1


In [15]:
df_acad_final.columns.tolist()

['Student_ID',
 'Year',
 'Period of Study',
 'APPLIED MECHANICS',
 'COMPUTER SKILLS',
 'CONSTRUCTION MATERIALS TECHNOLOGY',
 'ENGINEERING CHEMISTRY',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS 1',
 'ENGINEERING PHYSICS',
 'PROFESSIONAL AND TECHNICAL COMMUNICATION',
 'SURVEYING',
 'THEORY OF STRUCTURES']

In [16]:
# count distinct students per year.
df_acad_final.groupby("Year")["Student_ID"].nunique()

Year
2018     22
2019     43
2020     73
2021     63
2022     90
2023    151
2024    193
2025    219
2026     77
Name: Student_ID, dtype: int64

Enrolment For Civil Eng Students.

In [17]:
import pandas as pd
file_path = r"RE_ Enrolment data request\2022-2025_ Enrolment data_Civil_Eng.xlsx"

enrol_2022 =pd.read_excel(file_path, sheet_name="2022_Civil")
enrol_2023 =pd.read_excel(file_path, sheet_name="2023_Civil")
enrol_2024 =pd.read_excel(file_path, sheet_name="2024_Civil")
enrol_2025 =pd.read_excel(file_path, sheet_name="2025_Civil")
#enrol_2025 = pd.read_excel("FTE Dashboard and Data\RE_ Enrolment data request\Enrolments_2025_Civil.xlsx")

In [18]:
df_enrol = pd.concat([enrol_2022, enrol_2023, enrol_2024, enrol_2025], ignore_index=True)
#df_enrol.head()

In [19]:
df_enrol["Student_ID"] = df_enrol["Student Number"].apply(hash_student_id)


In [20]:
df_enrol.isnull().sum()

Academic Year                 0
Faculty School Name           0
Department Name               0
Student Number                0
First Names                   0
Surname                       0
Gender                        0
Language Name                 0
Ethnic Group Name             0
Country Name                  0
Birthdate                     0
Qualification Code            0
Qualification Name            0
Period Of Study               0
Ften Status This Year         0
Academic Block Code           0
Previous Activity Desc        0
Id Number                     0
Marital Status                0
Passport Number             466
Matric Type                   0
Student Type Description      0
School Quintile               3
Student_ID                    0
dtype: int64

In [21]:
df_enrol = df_enrol.dropna(axis=1, how="all")
df_enrol.isnull().sum()

Academic Year               0
Faculty School Name         0
Department Name             0
Student Number              0
First Names                 0
Surname                     0
Gender                      0
Language Name               0
Ethnic Group Name           0
Country Name                0
Birthdate                   0
Qualification Code          0
Qualification Name          0
Period Of Study             0
Ften Status This Year       0
Academic Block Code         0
Previous Activity Desc      0
Id Number                   0
Marital Status              0
Matric Type                 0
Student Type Description    0
School Quintile             3
Student_ID                  0
dtype: int64

In [22]:
df_enrol.columns.tolist()

['Academic Year',
 'Faculty School Name',
 'Department Name',
 'Student Number',
 'First Names',
 'Surname',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Qualification Code',
 'Qualification Name',
 'Period Of Study',
 'Ften Status This Year',
 'Academic Block Code',
 'Previous Activity Desc',
 'Id Number',
 'Marital Status',
 'Matric Type',
 'Student Type Description',
 'School Quintile',
 'Student_ID']

In [23]:
df_enrol.drop(columns= ['Student Number','First Names','Id Number', 'Surname','Qualification Code','Qualification Name'], inplace = True)

In [24]:
df_enrol["Student_ID"].nunique()

466

In [25]:
# count unique students for df_registration
df_enrol.groupby(["Academic Year"])["Student_ID"] \
    .nunique() \
    .reset_index(name="unique_students")

,Academic Year,unique_students
0,2022,100
1,2023,115
2,2024,134
3,2025,117


In [26]:
df_enrol["Student_ID"].nunique()

466

In [27]:
print("Enrolment", df_enrol.columns.tolist())
print("academic records", df_acad_final.columns.tolist())

Enrolment ['Academic Year', 'Faculty School Name', 'Department Name', 'Gender', 'Language Name', 'Ethnic Group Name', 'Country Name', 'Birthdate', 'Period Of Study', 'Ften Status This Year', 'Academic Block Code', 'Previous Activity Desc', 'Marital Status', 'Matric Type', 'Student Type Description', 'School Quintile', 'Student_ID']
academic records ['Student_ID', 'Year', 'Period of Study', 'APPLIED MECHANICS', 'COMPUTER SKILLS', 'CONSTRUCTION MATERIALS TECHNOLOGY', 'ENGINEERING CHEMISTRY', 'ENGINEERING DRAWING 1', 'ENGINEERING MATHEMATICS 1', 'ENGINEERING PHYSICS', 'PROFESSIONAL AND TECHNICAL COMMUNICATION', 'SURVEYING', 'THEORY OF STRUCTURES']


Combining Enrolment and academic records data

In [28]:
# 1. COPY DATA
enrol = df_enrol.copy()
acad = df_acad_final.copy()

# 2. STANDARDISE KEYS
enrol = enrol.rename(columns={
    "Academic Year": "Year"
})

enrol["Student_ID"] = enrol["Student_ID"].astype(str).str.strip()
acad["Student_ID"] = acad["Student_ID"].astype(str).str.strip()

enrol["Year"] = enrol["Year"].astype(str).str.strip()
acad["Year"] = acad["Year"].astype(str).str.strip()

# 3. FILTER Enrolment DATA
enrol = enrol[enrol["Period Of Study"] == 1]

# 4. CLEAN DATA

enrol = enrol.drop_duplicates(subset=["Student_ID", "Year"])
acad = acad.drop_duplicates(subset=["Student_ID", "Year"])

# 5. MERGE
merged_df = pd.merge(
    acad,
    enrol,
    on=["Student_ID", "Year"],
    how="inner"
)

# 6. CLEANUP
merged_df.columns = merged_df.columns.str.strip()

print("Final shape:", merged_df.shape)

Final shape: (395, 28)


In [29]:
merged_df.head()

,Student_ID,Year,Period of Study,APPLIED MECHANICS,COMPUTER SKILLS,CONSTRUCTION MATERIALS TECHNOLOGY,ENGINEERING CHEMISTRY,ENGINEERING DRAWING 1,ENGINEERING MATHEMATICS 1,ENGINEERING PHYSICS,...,Country Name,Birthdate,Period Of Study,Ften Status This Year,Academic Block Code,Previous Activity Desc,Marital Status,Matric Type,Student Type Description,School Quintile
0,8bafbd6d917a77865da3da45a93ffef06bcbb7f0001482...,2024,1,69.0,61.0,59.0,67.0,50.0,60.0,68.0,...,R.S.A.,1999-06-14,1,First Time Entering,0,GRADE12 LEARNER,S,B,South African Student,3.0
1,9d75f619004bb89e3125b25df1d963f3d4d0b0b48ef4d1...,2024,1,55.0,71.0,64.0,41.0,47.0,40.0,56.0,...,R.S.A.,2000-08-11,1,First Time Entering,0,GRADE12 LEARNER,S,B,South African Student,3.0
2,fce4a7e4e7c53b329ecad10a3c1b656b7295f14762efd5...,2023,1,82.0,90.0,81.0,85.0,83.0,91.0,75.0,...,R.S.A.,2001-03-26,1,First Time Entering,0,GRADE12 LEARNER,S,B,South African Student,4.0
3,14e415ee68561c55bfe7794d936a1e5c80050073ac8438...,2022,1,28.0,43.0,65.0,80.0,56.0,42.0,43.0,...,R.S.A.,2001-05-03,1,First Time Entering,0,OTHER,S,B,South African Student,1.0
4,a0e4763df4ec20f3e34dcddcff978b5b71d100c1c45827...,2024,1,59.0,76.0,82.0,43.0,53.0,56.0,53.0,...,R.S.A.,2001-06-13,1,First Time Entering,0,UNEMPLOYED,S,B,South African Student,3.0


In [30]:
merged_df = merged_df.drop(
    columns=["Cancellation Date", "Cancellation Description", "Approved Qualification Name","Period of Study", "Cancelled", "Qualification Type","Alien Indicator"],
    errors="ignore"
)

In [31]:
print(merged_df["Year"].dtype)
print(merged_df["Year"].head())

object
0    2024
1    2024
2    2023
3    2022
4    2024
Name: Year, dtype: object


In [32]:
merged_df["Year"] = pd.to_numeric(
    merged_df["Year"],
    errors="coerce"
)

In [33]:
# creating an Age_At_Enrolment column, using the birthdate col the Year.
# Convert Birthdate to datetime
merged_df["Birthdate"] = pd.to_datetime(merged_df["Birthdate"], errors="coerce")

# Extract birth year
merged_df["Birth_Year"] = merged_df["Birthdate"].dt.year

# Calculate age at enrolment
merged_df["Age_At_Enrolment"] = merged_df["Year"] - merged_df["Birth_Year"]

# Remove the temporary column
merged_df.drop(columns="Birth_Year", inplace=True)

In [34]:
merged_df.columns.tolist()

['Student_ID',
 'Year',
 'APPLIED MECHANICS',
 'COMPUTER SKILLS',
 'CONSTRUCTION MATERIALS TECHNOLOGY',
 'ENGINEERING CHEMISTRY',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS 1',
 'ENGINEERING PHYSICS',
 'PROFESSIONAL AND TECHNICAL COMMUNICATION',
 'SURVEYING',
 'THEORY OF STRUCTURES',
 'Faculty School Name',
 'Department Name',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Period Of Study',
 'Ften Status This Year',
 'Academic Block Code',
 'Previous Activity Desc',
 'Marital Status',
 'Matric Type',
 'Student Type Description',
 'School Quintile',
 'Age_At_Enrolment']

Finance Data

In [35]:
import pandas as pd

df_finance = pd.read_excel(
    r"Hemis_data\Hemis_data\FTE\Finance\Finance Civil_Eng (2022-2025).xlsx"
)

#df_finance.head()

In [36]:
df_finance.columns.tolist()

['StudentNo',
 'Academic Year',
 'Qualification Code',
 'Qualification Name',
 'Formal Time',
 'Bursary Type',
 'BURSARYCODE',
 'BURSARYNAME',
 'BURSARYFUND',
 'FUNDNAME']

In [37]:
df_finance["Student_ID"] = df_finance["StudentNo"].apply(hash_student_id)

In [38]:
df_finance.drop(columns= "StudentNo", inplace=True)

In [39]:
df_finance.columns.tolist()

['Academic Year',
 'Qualification Code',
 'Qualification Name',
 'Formal Time',
 'Bursary Type',
 'BURSARYCODE',
 'BURSARYNAME',
 'BURSARYFUND',
 'FUNDNAME',
 'Student_ID']

In [40]:
# drop bursasy type, bursarycode, bursary name, bursary fund, Fundname.
df_finance.drop(
    columns=[
        "BURSARYCODE",
        "BURSARYNAME",
        "BURSARYFUND",
        "FUNDNAME"
    ],
    inplace=True
)

df_finance.head()

,Academic Year,Qualification Code,Qualification Name,Formal Time,Bursary Type,Student_ID
0,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,47082b09ee3cafe8bd557655d1f09313975a4ba9b5b722...
1,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,67f05b7e851ef0a650288d9608341485189b445d6a72a9...
2,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,50aa4bc06baff9c6e4593dbe73a6d57ccb05b91049d2ee...
3,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,PRIVATE BURSARY,2fdae0754e821aeeefb29e8ceec025136ba8be00fb7b05...
4,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,fd09e12a24f65f914f4b0bf25d049af3d14818e4e82301...


In [41]:
df_finance.isnull().sum()

Academic Year         0
Qualification Code    0
Qualification Name    0
Formal Time           0
Bursary Type          0
Student_ID            0
dtype: int64

In [42]:
df_finance["Bursary Type"].value_counts()

Bursary Type
NSFAS              6994
Self-Funded         511
PRIVATE BURSARY     210
Name: count, dtype: int64

In [43]:
df_finance["Funded"] = df_finance["Bursary Type"].apply(
    lambda x: "No" if x == "Self-Funded" else "Yes"
)

df_finance["Funded"].value_counts()

Funded
Yes    7204
No      511
Name: count, dtype: int64

In [44]:
df_finance.groupby("Academic Year")["Student_ID"].nunique()

Academic Year
2022    339
2023    381
2024    408
2025    444
Name: Student_ID, dtype: int64

In [45]:
df_finance.head()

,Academic Year,Qualification Code,Qualification Name,Formal Time,Bursary Type,Student_ID,Funded
0,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,47082b09ee3cafe8bd557655d1f09313975a4ba9b5b722...,No
1,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,67f05b7e851ef0a650288d9608341485189b445d6a72a9...,No
2,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,50aa4bc06baff9c6e4593dbe73a6d57ccb05b91049d2ee...,No
3,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,PRIVATE BURSARY,2fdae0754e821aeeefb29e8ceec025136ba8be00fb7b05...,Yes
4,2022,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,fd09e12a24f65f914f4b0bf25d049af3d14818e4e82301...,No


In [46]:
df_finance.isnull().sum()

Academic Year         0
Qualification Code    0
Qualification Name    0
Formal Time           0
Bursary Type          0
Student_ID            0
Funded                0
dtype: int64

In [47]:
# count distinct students in finance data after filtering for years
df_finance["Student_ID"].nunique()

763

In [48]:
# check and give name of the distinct qualifications in finance data
df_finance["Qualification Code"].unique()



array(['D3CIVL'], dtype=object)

In [49]:
print(df_finance.columns.tolist())

['Academic Year', 'Qualification Code', 'Qualification Name', 'Formal Time', 'Bursary Type', 'Student_ID', 'Funded']


In [50]:
print(merged_df.columns.tolist())

['Student_ID', 'Year', 'APPLIED MECHANICS', 'COMPUTER SKILLS', 'CONSTRUCTION MATERIALS TECHNOLOGY', 'ENGINEERING CHEMISTRY', 'ENGINEERING DRAWING 1', 'ENGINEERING MATHEMATICS 1', 'ENGINEERING PHYSICS', 'PROFESSIONAL AND TECHNICAL COMMUNICATION', 'SURVEYING', 'THEORY OF STRUCTURES', 'Faculty School Name', 'Department Name', 'Gender', 'Language Name', 'Ethnic Group Name', 'Country Name', 'Birthdate', 'Period Of Study', 'Ften Status This Year', 'Academic Block Code', 'Previous Activity Desc', 'Marital Status', 'Matric Type', 'Student Type Description', 'School Quintile', 'Age_At_Enrolment']


In [51]:
merged_df = merged_df.rename(columns={
    "Year": "Academic Year"
})

Finance, Academic Records and enrolment data Merge

In [52]:
print(merged_df.dtypes["Academic Year"])
print(df_finance.dtypes["Academic Year"])

int64
int64


In [53]:
merged_df["Academic Year"] = merged_df["Academic Year"].astype(str).str.strip()
df_finance["Academic Year"] = df_finance["Academic Year"].astype(str).str.strip()

In [54]:
df_finance_clean = (
    df_finance
    .sort_values(['Student_ID', 'Academic Year'])
    .drop_duplicates(subset=['Student_ID', 'Academic Year'], keep='first')
)

df_combined = merged_df.merge(
    df_finance_clean,
    on=['Student_ID', 'Academic Year'],
    how='left'
)

# Check duplicates
print("Duplicate rows:", df_combined.duplicated().sum())

Duplicate rows: 0


In [55]:
# Keep the earliest first-year record for each student
df_combined = (
    df_combined
    .sort_values("Academic Year")
    .drop_duplicates(subset="Student_ID", keep="first")
    .reset_index(drop=True)
)

In [56]:
df_combined.columns

Index(['Student_ID', 'Academic Year', 'APPLIED MECHANICS', 'COMPUTER SKILLS',
       'CONSTRUCTION MATERIALS TECHNOLOGY', 'ENGINEERING CHEMISTRY',
       'ENGINEERING DRAWING 1', 'ENGINEERING MATHEMATICS 1',
       'ENGINEERING PHYSICS', 'PROFESSIONAL AND TECHNICAL COMMUNICATION',
       'SURVEYING', 'THEORY OF STRUCTURES', 'Faculty School Name',
       'Department Name', 'Gender', 'Language Name', 'Ethnic Group Name',
       'Country Name', 'Birthdate', 'Period Of Study', 'Ften Status This Year',
       'Academic Block Code', 'Previous Activity Desc', 'Marital Status',
       'Matric Type', 'Student Type Description', 'School Quintile',
       'Age_At_Enrolment', 'Qualification Code', 'Qualification Name',
       'Formal Time', 'Bursary Type', 'Funded'],
      dtype='object')

In [57]:
# check for unique count on df_combined
df_combined["Student_ID"].nunique()

395

In [58]:
df_combined.columns.tolist()

['Student_ID',
 'Academic Year',
 'APPLIED MECHANICS',
 'COMPUTER SKILLS',
 'CONSTRUCTION MATERIALS TECHNOLOGY',
 'ENGINEERING CHEMISTRY',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS 1',
 'ENGINEERING PHYSICS',
 'PROFESSIONAL AND TECHNICAL COMMUNICATION',
 'SURVEYING',
 'THEORY OF STRUCTURES',
 'Faculty School Name',
 'Department Name',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Period Of Study',
 'Ften Status This Year',
 'Academic Block Code',
 'Previous Activity Desc',
 'Marital Status',
 'Matric Type',
 'Student Type Description',
 'School Quintile',
 'Age_At_Enrolment',
 'Qualification Code',
 'Qualification Name',
 'Formal Time',
 'Bursary Type',
 'Funded']

In [59]:
df_combined.isnull().sum()

Student_ID                                   0
Academic Year                                0
APPLIED MECHANICS                            2
COMPUTER SKILLS                              3
CONSTRUCTION MATERIALS TECHNOLOGY            3
ENGINEERING CHEMISTRY                        2
ENGINEERING DRAWING 1                        8
ENGINEERING MATHEMATICS 1                    2
ENGINEERING PHYSICS                          0
PROFESSIONAL AND TECHNICAL COMMUNICATION     5
SURVEYING                                    3
THEORY OF STRUCTURES                        58
Faculty School Name                          0
Department Name                              0
Gender                                       0
Language Name                                0
Ethnic Group Name                            0
Country Name                                 0
Birthdate                                    0
Period Of Study                              0
Ften Status This Year                        0
Academic Bloc

Residence Data

In [60]:
import pandas as pd

# Read the 2022 residence data
df_res_2022 = pd.read_excel(
    r"Hemis_data\Hemis_data\FTE\Residence\residence_data2022_Civil.xlsx"
)

# Read the 2023-2025 residence data
df_res_2023_2025 = pd.read_excel(
    r"Hemis_data\Hemis_data\FTE\Residence\Residence 2023_2025_Civil.xlsx"
)

df_residence = pd.concat(
    [df_res_2022, df_res_2023_2025],
    ignore_index=True
)

print(df_residence.shape)
#df_residence.head()

(229, 13)


In [61]:
df_residence["Student_ID"] = df_residence["STUDENT"].apply(hash_student_id)

In [62]:
df_residence.isnull().sum()

STUDENT            0
SURNAME            0
FIRST_NAMES        0
TITLE              0
GENDER             0
BIRTH_DATE         0
AGE                0
ACADEMIC_YEAR      0
FACULTY_NAME       0
DEPARTMENT_NAME    0
QUALIFICATION      0
PERIOD_STUDY       0
FTEN_STATUS        0
Student_ID         0
dtype: int64

In [63]:
df_residence.columns.tolist()

['STUDENT',
 'SURNAME',
 'FIRST_NAMES',
 'TITLE',
 'GENDER',
 'BIRTH_DATE',
 'AGE',
 'ACADEMIC_YEAR',
 'FACULTY_NAME',
 'DEPARTMENT_NAME',
 'QUALIFICATION',
 'PERIOD_STUDY',
 'FTEN_STATUS',
 'Student_ID']

In [64]:
df_residence.drop(columns=['STUDENT','SURNAME','FIRST_NAMES','TITLE'], inplace=True)

In [65]:
# 1. COPY DATA
df_combined = df_combined.copy()
res = df_residence.copy()

# 3. STANDARDISE DUPLICATE COLUMN NAMES
df_combined["Student_ID"] = df_combined["Student_ID"].astype(str).str.strip()
res["Student_ID"] = res["Student_ID"].astype(str).str.strip()

# Secondary school standardisation (optional alignment)
res = res.rename(columns={
    "SECONDARY_SCHOOL_NAME": "Secondary_School_Name"
})

# 4. CREATE RESIDENCE FLAG
res["Residence_Status"] = "Yes"

res_flags = res[["Student_ID", "Residence_Status"]].drop_duplicates()

# 5. MERGE DATASETS
final_df = df_combined.merge(
    res_flags,
    on="Student_ID",
    how="left"
)

# Fill missing values
final_df["Residence_Status"] = final_df["Residence_Status"].fillna("No")

# 6. RESULT CHECK
print(final_df["Residence_Status"].value_counts())

final_df.head()

Residence_Status
Yes    202
No     193
Name: count, dtype: int64


,Student_ID,Academic Year,APPLIED MECHANICS,COMPUTER SKILLS,CONSTRUCTION MATERIALS TECHNOLOGY,ENGINEERING CHEMISTRY,ENGINEERING DRAWING 1,ENGINEERING MATHEMATICS 1,ENGINEERING PHYSICS,PROFESSIONAL AND TECHNICAL COMMUNICATION,...,Matric Type,Student Type Description,School Quintile,Age_At_Enrolment,Qualification Code,Qualification Name,Formal Time,Bursary Type,Funded,Residence_Status
0,ebad61ae15967effac440603214e2e829bb29399e52236...,2022,80.0,76.0,67.0,77.0,67.0,62.0,50.0,58.0,...,B,South African Student,3.0,19,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,Yes
1,f3ed539337ba8583af0b544e8cedad7f2066a7a311d761...,2022,NaN,52.0,58.0,75.0,54.0,66.0,50.0,53.0,...,B,South African Student,3.0,18,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,No
2,f521b30f9daaa32d07004bfe337461cbe32083db1a4c4f...,2022,68.0,42.0,62.0,72.0,65.0,66.0,55.0,63.0,...,B,South African Student,2.0,19,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,No
3,a39ac7e2e46a3c504ea680c833542b44509590f9566861...,2022,50.0,59.0,63.0,69.0,73.0,51.0,50.0,62.0,...,B,South African Student,5.0,20,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,Yes
4,32d8a74e94211a616b9c701f62ebf8ef4a90162ab430a6...,2022,56.0,62.0,62.0,75.0,55.0,50.0,53.0,57.0,...,B,South African Student,1.0,20,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,Yes


In [66]:
final_df.columns.tolist()

['Student_ID',
 'Academic Year',
 'APPLIED MECHANICS',
 'COMPUTER SKILLS',
 'CONSTRUCTION MATERIALS TECHNOLOGY',
 'ENGINEERING CHEMISTRY',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS 1',
 'ENGINEERING PHYSICS',
 'PROFESSIONAL AND TECHNICAL COMMUNICATION',
 'SURVEYING',
 'THEORY OF STRUCTURES',
 'Faculty School Name',
 'Department Name',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Period Of Study',
 'Ften Status This Year',
 'Academic Block Code',
 'Previous Activity Desc',
 'Marital Status',
 'Matric Type',
 'Student Type Description',
 'School Quintile',
 'Age_At_Enrolment',
 'Qualification Code',
 'Qualification Name',
 'Formal Time',
 'Bursary Type',
 'Funded',
 'Residence_Status']

In [67]:
df= final_df.copy()

In [68]:
df

,Student_ID,Academic Year,APPLIED MECHANICS,COMPUTER SKILLS,CONSTRUCTION MATERIALS TECHNOLOGY,ENGINEERING CHEMISTRY,ENGINEERING DRAWING 1,ENGINEERING MATHEMATICS 1,ENGINEERING PHYSICS,PROFESSIONAL AND TECHNICAL COMMUNICATION,...,Matric Type,Student Type Description,School Quintile,Age_At_Enrolment,Qualification Code,Qualification Name,Formal Time,Bursary Type,Funded,Residence_Status
0,ebad61ae15967effac440603214e2e829bb29399e52236...,2022,80.0,76.0,67.0,77.0,67.0,62.0,50.0,58.0,...,B,South African Student,3.0,19,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,Yes
1,f3ed539337ba8583af0b544e8cedad7f2066a7a311d761...,2022,NaN,52.0,58.0,75.0,54.0,66.0,50.0,53.0,...,B,South African Student,3.0,18,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,No
2,f521b30f9daaa32d07004bfe337461cbe32083db1a4c4f...,2022,68.0,42.0,62.0,72.0,65.0,66.0,55.0,63.0,...,B,South African Student,2.0,19,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,No
3,a39ac7e2e46a3c504ea680c833542b44509590f9566861...,2022,50.0,59.0,63.0,69.0,73.0,51.0,50.0,62.0,...,B,South African Student,5.0,20,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,Yes
4,32d8a74e94211a616b9c701f62ebf8ef4a90162ab430a6...,2022,56.0,62.0,62.0,75.0,55.0,50.0,53.0,57.0,...,B,South African Student,1.0,20,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,526bb8b3e15d5bef0811e6b6087ce1da9a3ba900d3ceef...,2025,50.0,30.0,66.0,59.0,39.0,37.0,57.0,79.0,...,B,South African Student,1.0,20,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,No
391,291b1a79921cda4db10a182d9b1b2f0a9741b1524bdee6...,2025,46.0,13.0,19.0,41.0,24.0,5.0,16.0,54.0,...,D,South African Student,1.0,27,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,Self-Funded,No,Yes
392,666eaec5b4eb2d73b50fdf78b644d950152195bc33143f...,2025,62.0,80.0,58.0,68.0,55.0,68.0,58.0,76.0,...,B,South African Student,1.0,20,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,No
393,00defb26a8ce72e2c2c8600b5c673494bb9d9cb4aff6d8...,2025,77.0,71.0,56.0,63.0,57.0,63.0,62.0,75.0,...,B,South African Student,2.0,21,D3CIVL,DIPLOMA IN CIVIL ENGINEERING,3,NSFAS,Yes,Yes


In [69]:
df.columns.tolist()

['Student_ID',
 'Academic Year',
 'APPLIED MECHANICS',
 'COMPUTER SKILLS',
 'CONSTRUCTION MATERIALS TECHNOLOGY',
 'ENGINEERING CHEMISTRY',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS 1',
 'ENGINEERING PHYSICS',
 'PROFESSIONAL AND TECHNICAL COMMUNICATION',
 'SURVEYING',
 'THEORY OF STRUCTURES',
 'Faculty School Name',
 'Department Name',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Period Of Study',
 'Ften Status This Year',
 'Academic Block Code',
 'Previous Activity Desc',
 'Marital Status',
 'Matric Type',
 'Student Type Description',
 'School Quintile',
 'Age_At_Enrolment',
 'Qualification Code',
 'Qualification Name',
 'Formal Time',
 'Bursary Type',
 'Funded',
 'Residence_Status']

In [70]:
MODULES = [
    'APPLIED MECHANICS',
    'CONSTRUCTION MATERIALS TECHNOLOGY',
    'COMPUTER SKILLS',
    'ENGINEERING DRAWING 1',
    'ENGINEERING CHEMISTRY',
    'ENGINEERING PHYSICS',
    'ENGINEERING MATHEMATICS 1',
    'PROFESSIONAL AND TECHNICAL COMMUNICATION',
    'SURVEYING',
    'THEORY OF STRUCTURES'
]

# Count how many modules each student is registered for
# (assuming non-null values mean registered)
student_module_counts = df[MODULES].notna().sum(axis=1)

# Create frequency table of how many students have each number of registrations
reg_distribution = student_module_counts.value_counts().sort_index().reset_index()
reg_distribution.columns = ['Number of Modules', 'Number of Students']

# Add percentage column
reg_distribution['Percentage (%)'] = round(
    (reg_distribution['Number of Students'] / len(df)) * 100, 2
)

print(f"Total students in dataset: {len(df)}")
print()
print(reg_distribution.to_string(index=False))

# Add cumulative statistics
print()
print("Summary Statistics:")
print(f"Mean modules per student: {student_module_counts.mean():.2f}")
print(f"Median modules per student: {student_module_counts.median():.0f}")
print(f"Min modules registered: {student_module_counts.min()}")
print(f"Max modules registered: {student_module_counts.max()}")

Total students in dataset: 395

 Number of Modules  Number of Students  Percentage (%)
                 4                   1            0.25
                 6                   2            0.51
                 7                   2            0.51
                 8                   7            1.77
                 9                  52           13.16
                10                 331           83.80

Summary Statistics:
Mean modules per student: 9.78
Median modules per student: 10
Min modules registered: 4
Max modules registered: 10


In [71]:
# 1. DEFINE SUBJECT COLUMNS
subject_cols = [
    'APPLIED MECHANICS',
    'COMPUTER SKILLS',
    'CONSTRUCTION MATERIALS TECHNOLOGY',
    'ENGINEERING CHEMISTRY',
    'ENGINEERING DRAWING 1',
    'ENGINEERING MATHEMATICS 1',
    'ENGINEERING PHYSICS',
    'PROFESSIONAL AND TECHNICAL COMMUNICATION',
    'SURVEYING',
    'THEORY OF STRUCTURES'
]

# 2. DROP STUDENTS WITH ANY MISSING SUBJECT (i.e., not registered for all 10)
df = df.dropna(subset=subject_cols, how='any')

# 4. CREATE AT-RISK COLUMN
df["At_Risk_Status"] = (
    (df[subject_cols] < 50).any(axis=1)
).map({True: "Yes", False: "No"})

print(df["At_Risk_Status"].value_counts())

At_Risk_Status
No     209
Yes    122
Name: count, dtype: int64


C:\Users\bayibayik\AppData\Local\Temp\ipykernel_39912\12254517.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["At_Risk_Status"] = (


In [72]:
df.dtypes

Student_ID                                          object
Academic Year                                       object
APPLIED MECHANICS                                  float64
COMPUTER SKILLS                                    float64
CONSTRUCTION MATERIALS TECHNOLOGY                  float64
ENGINEERING CHEMISTRY                              float64
ENGINEERING DRAWING 1                              float64
ENGINEERING MATHEMATICS 1                          float64
ENGINEERING PHYSICS                                float64
PROFESSIONAL AND TECHNICAL COMMUNICATION           float64
SURVEYING                                          float64
THEORY OF STRUCTURES                               float64
Faculty School Name                                 object
Department Name                                     object
Gender                                              object
Language Name                                       object
Ethnic Group Name                                   obje

In [73]:
# drop identifier columns such as Student_Number, Names, Surname, Title, Initials, and any other columns that may identify a student
df = df.drop(columns=[
    "StudentNo",
    "STUDENT",
    "First Names",
    "Surname",
    "Title",
    "Initials"
], errors="ignore")

# drop Formal Time, Campus Name, Qualification Name.
df = df.drop(columns=[
    "Campus_Name", 
    "Formal Time",
    "Faculty School Name",
    "Department Name",
    "Qualification Code",
    "Qualification Name",
    "Birthdate"
], errors="ignore")

print("Final columns:", df.columns.tolist())


Final columns: ['Student_ID', 'Academic Year', 'APPLIED MECHANICS', 'COMPUTER SKILLS', 'CONSTRUCTION MATERIALS TECHNOLOGY', 'ENGINEERING CHEMISTRY', 'ENGINEERING DRAWING 1', 'ENGINEERING MATHEMATICS 1', 'ENGINEERING PHYSICS', 'PROFESSIONAL AND TECHNICAL COMMUNICATION', 'SURVEYING', 'THEORY OF STRUCTURES', 'Gender', 'Language Name', 'Ethnic Group Name', 'Country Name', 'Period Of Study', 'Ften Status This Year', 'Academic Block Code', 'Previous Activity Desc', 'Marital Status', 'Matric Type', 'Student Type Description', 'School Quintile', 'Age_At_Enrolment', 'Bursary Type', 'Funded', 'Residence_Status', 'At_Risk_Status']


In [74]:
df.dtypes

Student_ID                                   object
Academic Year                                object
APPLIED MECHANICS                           float64
COMPUTER SKILLS                             float64
CONSTRUCTION MATERIALS TECHNOLOGY           float64
ENGINEERING CHEMISTRY                       float64
ENGINEERING DRAWING 1                       float64
ENGINEERING MATHEMATICS 1                   float64
ENGINEERING PHYSICS                         float64
PROFESSIONAL AND TECHNICAL COMMUNICATION    float64
SURVEYING                                   float64
THEORY OF STRUCTURES                        float64
Gender                                       object
Language Name                                object
Ethnic Group Name                            object
Country Name                                 object
Period Of Study                               int64
Ften Status This Year                        object
Academic Block Code                           int64
Previous Act

In [75]:
df.isnull().sum()

Student_ID                                  0
Academic Year                               0
APPLIED MECHANICS                           0
COMPUTER SKILLS                             0
CONSTRUCTION MATERIALS TECHNOLOGY           0
ENGINEERING CHEMISTRY                       0
ENGINEERING DRAWING 1                       0
ENGINEERING MATHEMATICS 1                   0
ENGINEERING PHYSICS                         0
PROFESSIONAL AND TECHNICAL COMMUNICATION    0
SURVEYING                                   0
THEORY OF STRUCTURES                        0
Gender                                      0
Language Name                               0
Ethnic Group Name                           0
Country Name                                0
Period Of Study                             0
Ften Status This Year                       0
Academic Block Code                         0
Previous Activity Desc                      0
Marital Status                              0
Matric Type                       

In [76]:
df["Secondary_School_Quintile"] = df["School Quintile"].astype("Int64")

In [77]:
# check for the unique values in Secondary_School_Quintile.
df["Secondary_School_Quintile"].unique()

<IntegerArray>
[3, 2, 5, 1, 4, <NA>]
Length: 6, dtype: Int64

In [78]:
# compare Period of Study and Period_Of_Study columns in the dataframe
if "Period of Study" in df.columns and "Period_Of_Study" in df.columns:
    period_of_study_diff = df["Period of Study"].compare(df["Period_Of_Study"])
    print("Differences between 'Period of Study' and 'Period_Of_Study':")
    print(period_of_study_diff) 



In [79]:
# drop one of the Period of Study columns to avoid redundancy
if "Period of Study" in df.columns:
    df = df.drop(columns=["Period of Study"])

    

In [80]:
df.isnull().sum()

Student_ID                                  0
Academic Year                               0
APPLIED MECHANICS                           0
COMPUTER SKILLS                             0
CONSTRUCTION MATERIALS TECHNOLOGY           0
ENGINEERING CHEMISTRY                       0
ENGINEERING DRAWING 1                       0
ENGINEERING MATHEMATICS 1                   0
ENGINEERING PHYSICS                         0
PROFESSIONAL AND TECHNICAL COMMUNICATION    0
SURVEYING                                   0
THEORY OF STRUCTURES                        0
Gender                                      0
Language Name                               0
Ethnic Group Name                           0
Country Name                                0
Period Of Study                             0
Ften Status This Year                       0
Academic Block Code                         0
Previous Activity Desc                      0
Marital Status                              0
Matric Type                       

In [81]:
df["Previous Activity Desc"].unique()

array(['UNEMPLOYED', 'OTHER', 'GRADE12 LEARNER', 'UNIVERSITY STUDENT',
       'TECHNICAL COLLEGE / TVET', 'ENROLLED - FOREIGN INSTITUTION',
       'LABOUR FORCE'], dtype=object)

In [82]:
df = df.drop(columns=["School Quintile", "Period Of Study", "Country Name","Ften Status This Year","Academic Block Code", "Marital Status", "Student Type Description"])

In [83]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

missing

Secondary_School_Quintile    2
dtype: int64

In [84]:
missing_percent = (df.isnull().mean()*100).round(2)

missing_percent[missing_percent > 0]

Secondary_School_Quintile    0.6
dtype: float64

In [85]:
print(df['Academic Year'].dtype)
print(df['Academic Year'].unique())

object
['2022' '2023' '2024' '2025']


In [86]:
df.columns.tolist()

['Student_ID',
 'Academic Year',
 'APPLIED MECHANICS',
 'COMPUTER SKILLS',
 'CONSTRUCTION MATERIALS TECHNOLOGY',
 'ENGINEERING CHEMISTRY',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS 1',
 'ENGINEERING PHYSICS',
 'PROFESSIONAL AND TECHNICAL COMMUNICATION',
 'SURVEYING',
 'THEORY OF STRUCTURES',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Previous Activity Desc',
 'Matric Type',
 'Age_At_Enrolment',
 'Bursary Type',
 'Funded',
 'Residence_Status',
 'At_Risk_Status',
 'Secondary_School_Quintile']

Handling Missing Values.

In [87]:
df.isnull().sum()

Student_ID                                  0
Academic Year                               0
APPLIED MECHANICS                           0
COMPUTER SKILLS                             0
CONSTRUCTION MATERIALS TECHNOLOGY           0
ENGINEERING CHEMISTRY                       0
ENGINEERING DRAWING 1                       0
ENGINEERING MATHEMATICS 1                   0
ENGINEERING PHYSICS                         0
PROFESSIONAL AND TECHNICAL COMMUNICATION    0
SURVEYING                                   0
THEORY OF STRUCTURES                        0
Gender                                      0
Language Name                               0
Ethnic Group Name                           0
Previous Activity Desc                      0
Matric Type                                 0
Age_At_Enrolment                            0
Bursary Type                                0
Funded                                      0
Residence_Status                            0
At_Risk_Status                    

In [88]:
# change Secondary_School_Quintile to numeric
df["Secondary_School_Quintile"] = pd.to_numeric(df["Secondary_School_Quintile"], errors="coerce")


In [89]:
# Convert to object
df["Secondary_School_Quintile"] = df["Secondary_School_Quintile"].astype("object")

# Calculate mode
mode = df["Secondary_School_Quintile"].mode()[0]

# Replace missing values with mode
df["Secondary_School_Quintile"] = df["Secondary_School_Quintile"].fillna(mode)

# Check
print("Mode used:", mode)
print(df["Secondary_School_Quintile"].dtype)
print(df["Secondary_School_Quintile"].value_counts(dropna=False))

Mode used: 5
int64
Secondary_School_Quintile
5    102
3     70
2     67
1     60
4     32
Name: count, dtype: int64


C:\Users\bayibayik\AppData\Local\Temp\ipykernel_39912\224340455.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Secondary_School_Quintile"] = df["Secondary_School_Quintile"].fillna(mode)


In [90]:
print(df["Gender"].value_counts(dropna=False))

Gender
M    180
F    150
U      1
Name: count, dtype: int64


In [91]:
# drop student whose Gender is U.
df = df[df["Gender"] != "U"]

In [92]:
print(df["Gender"].value_counts(dropna=False))

Gender
M    180
F    150
Name: count, dtype: int64


In [93]:
df['Language Name'] = df['Language Name'].replace({
    'SESOTHO(SOUTH SOTHO)':              'SESOTHO',
    'SESOTHO SA LEBOA (NORTH SOTHO)':    'SESOTHO',
    'AFRIKAANS/ENGLISH':                 'ENGLISH',
})

In [94]:
print(df["Language Name"].value_counts(dropna=False))

Language Name
XHOSA         129
ISIZULU        68
ENGLISH        32
SETSWANA       24
SEPEDI         23
SESOTHO        17
AFRIKAANS      11
SISWATI         8
XITSONGA        8
TSHIVENDA       7
ISINDEBELE      3
Name: count, dtype: int64


In [95]:
# save df as an excel file
df.to_excel("df_final.xlsx", index=False)